In [ ]:
#| default_exp machine_learning.context_extraction

In [ ]:
#| export
from typing import Optional

import lmstudio as lms
from lmstudio import LLM

from trouver.notation.glossary import _resolve_info_notes_for_index
from trouver.obsidian.vault import VaultNote
from trouver.obsidian.file import MarkdownFile
from trouver.personal_vault.note_processing import process_standard_information_note

In [ ]:
#| export

# Assuming the client is imported from your established lmstudio module
# from lmstudio.client import LMStudio 

# --- Constants for the LLM ---
CONTEXT_EXTRACTION_SYSTEM_PROMPT = r"""
### Role
You are a Mathematical Secretary and Pre-Processor. Your goal is to read mathematical prose and extract the "Ambient Environment" before any formal statements (Propositions/Theorems) are encountered. You are preparing a "Silver Platter" of definitions so that a reader can understand future formal statements without looking back. You will be provided with a 'Previous Context' (existing definitions) and a 'New Passage'. Your task is to update the 'Silver Platter' by identifying only what has changed or been added.

### Extraction Focus
1. **Persistent Notational Conventions**: Identify ONLY symbols introduced for the first time. If a symbol exists in 'Previous Context', it is forbidden to list it under 'Primary Objects'.
2. **Ambient Assumptions**: Extract the new properties assigned to the objects currently under discussion (e.g., "assume all manifolds are smooth," "let $X$ be a compact space").
3. **Strict Non-Redundancy**: If a passage provides new information about an object already in 'Previous Context' (e.g., "K is now assumed to be finite"), move this information to Global Constraints or Defined Relations. Do not re-instantiate the object.
4. **Implicit Structural Relations**: Identify how objects relate to each other as established in the prose (e.g., $A \subseteq B$ is a ring extension, $G$ acts on $X$).
5. **Scope of Validity**: Note if the author explicitly limits the discussion (e.g., "Throughout this section, we assume $n > 2$").
6. The Null Case: If the passage contains no new notations or assumptions beyond what is in the 'Previous Context', output the 'Status' message below. If the passage is purely expository, motivational, or repeats 'Previous Context' without adding new constraints/notations, you must only output the Status message. Do not include empty headers or "None new" lists.

### Input Structure
You will receive input in the following format:
* **PREVIOUS CONTEXT**: A summary of mathematical objects and notations already established.
* **NEW PASSAGE**: The raw text to be analyzed for new additions.

### Output Format (The Silver Platter)
If the text contains relevant information, use the following structure:

**New Contextual Updates:**
* **Primary Objects**: [e.g., $K$ (Algebraic number field), $V$ (Vector space)]
* **Defined Relations**: [e.g., $A$ is a subring of $B$]
* **Active Notations**: [e.g., $S$ = set of prime ideals, $Cl^S_K$ = S-class group]
* **Global Constraints**: [e.g., "All rings are assumed to be Noetherian," "Characteristic of $K$ is 0"]

**If no relevant information is found:**
* **Status**: No new contextual or notational instantiations identified in this passage.

Only include a bullet point if there is information to populate it. If a specific section (e.g., Active Notations) has no updates but others do, omit that bullet point entirely.

---
### Technical Directives
* Prioritize the accuracy of symbol extraction over lengthy prose explanations.
* Use LaTeX for all mathematical symbols. Generally maintain the LaTeX convention of the original text except to fix typos in the original text.
* Prohibit Meta-talk: Do not include "Notes," "Explanations," or parenthetical justifications (e.g., "(already defined)"). The Silver Platter must contain only the mathematical data.
* Property Mapping: Treat adjectives (e.g., "compact," "Noetherian," "flat") as Global Constraints or Defined Relations, never as Primary Objects.
"""

# def extract_context_with_lm(
#         model: LLM,
#         excerpt_text: str,
#         # max_context: int = 4096,
#         system_prompt: str = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
#         ) -> Optional[str]:
#     """
#     Uses a local LM Studio model to extract contextual information from text.
#     This implementation uses the project's specific `lmstudio.client.LMStudio` class.

#     Args:
#         text: The mathematical text to analyze.
#         model_name: The specific model to use for the completion (e.g., 'local-model').

#     Returns:
#         The string output from the model, or None if an error occurs.
#     """
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": excerpt_text},
#     ]

#     try:
#         # Initialize the client as per the pattern in your project
#         result = model.respond(
#             {"messages": messages}, 
#             config={"temperature": 0.1}
#         )
        
#         # 3. Access .content, don't just stringify the object
#         if hasattr(result, 'content'):
#             return result.content.strip()
#         return str(result).strip()
#     except Exception as e:
#         # This generic exception can be refined if your lmstudio client has specific errors
#         print(f"An error occurred while communicating with LM Studio: {e}")
#         return None

In [ ]:
#| export
# In context_extraction module
from trouver.llm_core.call_llm import call_llm, process_llm_response, smart_truncate, SupportedLLM
def extract_context_with_lm(
    model: SupportedLLM,
    excerpt_text: str,
    max_context: int = 4096,
    system_prompt: str = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
    config: Optional[dict] = None,
    verbose: bool = True
) -> Optional[str]:
    
    # Context extraction usually needs more 'room' for the input text
    reserved = 1200 
    truncated_text = smart_truncate(excerpt_text, model, max_context, reserved)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": truncated_text}
    ]

    raw_output = call_llm(model, messages, config, verbose)
    # We use the 'clean' version in case the model starts thinking
    return process_llm_response(raw_output, return_thoughts=False)

In [ ]:
#| export
from typing import List, Optional
from pathlib import Path


# --- Assumed Imports from Your Library ---
#
#
#   from .glossary import _resolve_info_notes_for_index

def _generate_context_extraction_markdown(
        info_notes: List[VaultNote],
        model: LLM,
        system_prompt: str,
        ) -> str:
    """
    Generates Markdown by extracting context from a list of info notes.
    """
    markdown_lines = []
    for note in info_notes:
        print(f"  - Processing '{note.name}' for context...")
        mf = MarkdownFile.from_vault_note(note)
        note_text = str(process_standard_information_note(mf, note.vault))
        
        # This call now correctly uses your internal lmstudio library via the updated function
        context_info = extract_context_with_lm(
            model,
            note_text, 
        )
        
        if context_info:
            markdown_lines.append(f"- [[{note.name}]]")
            context_lines = context_info.split('\n')
            markdown_lines.append(f"    - {context_lines[0]}")
            for line in context_lines[1:]:
                markdown_lines.append(f"      {line}")
                
    return "\n".join(markdown_lines)

In [ ]:
#| export
import re
from typing import List, Optional, Dict
from pathlib import Path

# --- Assumed Imports from Your Library ---
# from trouver.obsidian.vault import VaultNote
# from trouver.obsidian.markdown.file import MarkdownFile
# from trouver.obsidian.markdown.processing import process_standard_information_note
# from .glossary import _resolve_info_notes_for_index
# from .llm import extract_context_with_lm, LLM, CONTEXT_EXTRACTION_SYSTEM_PROMPT

def _parse_existing_context_note(note_text: str) -> Dict[str, str]:
    """
    Parses an existing context extraction note into a dictionary.
    
    Args:
        note_text: The content of the markdown note.
        
    Returns:
        A dictionary where keys are note names and values are the context strings.
    """
    data = {}
    current_note_name = None
    current_lines = []
    
    # Regex to match "- [[Note Name]]"
    note_header_pattern = re.compile(r'^\s*-\s*\[\[(.*?)\]\]')
    
    for line in note_text.split('\n'):
        match = note_header_pattern.match(line)
        if match:
            # If we were processing a note, save it before starting the new one
            if current_note_name:
                data[current_note_name] = "\n".join(current_lines).strip()
            
            current_note_name = match.group(1)
            current_lines = []
        elif current_note_name:
            # This line belongs to the current note.
            # We strip the list formatting (indentation + dash) to get the raw text.
            # Assuming format: "    - Text" or "      Text"
            stripped = line.strip()
            if stripped.startswith('- '):
                stripped = stripped[2:]
            current_lines.append(stripped)
            
    # Save the last note
    if current_note_name:
        data[current_note_name] = "\n".join(current_lines).strip()
        
    return data

def _format_context_extraction(
    info_notes: List[VaultNote], 
    context_data: Dict[str, str]
) -> str:
    """
    Formats the context dictionary back into the Markdown list structure,
    preserving the order of the provided info_notes list.
    """
    markdown_lines = []
    for note in info_notes:
        if note.name in context_data:
            context_text = context_data[note.name]
            if context_text:
                markdown_lines.append(f"- [[{note.name}]]")
                # Split multi-line context to format correctly
                lines = context_text.split('\n')
                # First line gets the bullet
                markdown_lines.append(f"    - {lines[0]}")
                # Subsequent lines get indentation
                for line in lines[1:]:
                    markdown_lines.append(f"      {line}")
    return "\n".join(markdown_lines)


In [ ]:
#| export

def _format_llm_input(previous_context: str, new_passage: str) -> str:
    """Formats the input for the LLM with previous context and new text."""
    # If there is no previous context, we can indicate that or leave it empty.
    context_str = previous_context if previous_context.strip() else "None."
    return f"PREVIOUS CONTEXT:\n{context_str}\n\nNEW PASSAGE:\n{new_passage}"

In [ ]:
#| export
import re
from typing import List, Optional, Dict, Callable
from pathlib import Path

# --- Assumed Imports ---
# from trouver.obsidian.vault import VaultNote
# from trouver.obsidian.markdown.file import MarkdownFile
# from trouver.obsidian.markdown.processing import process_standard_information_note
# from .glossary import _resolve_info_notes_for_index
# from .llm import extract_context_with_lm, LLM

# Updated Type Alias
# Inputs: 
#   1. The current VaultNote
#   2. The dictionary of all existing context data
#   3. The ordered list of all note names in the sequence
# Output: A list of note names (keys in the dictionary) to include
ContextSelector = Callable[[VaultNote, Dict[str, str], List[str]], List[str]]

def _default_context_selector(
    note: VaultNote, 
    current_data: Dict[str, str],
    ordered_note_names: List[str]
) -> List[str]:
    """
    Default behavior: Returns the context of the 5 most recent notes 
    that appear before the current note in the ordered list.
    """
    try:
        current_index = ordered_note_names.index(note.name)
    except ValueError:
        # Fallback: if note not found in list, return everything (unlikely)
        return [n for n in current_data.keys() if n != note.name]

    # Get all potential predecessors
    predecessors = ordered_note_names[:current_index]
    
    # Filter to keep only those that actually have data in current_data
    available_predecessors = [name for name in predecessors if name in current_data]
    
    # Return the last 5 (the most recent ones)
    return available_predecessors[-5:]


In [ ]:

#| export
def create_context_extraction_for_index_note(
    index_note: VaultNote,
    model: 'LLM',
    info_notes: Optional[List[VaultNote]] = None,
    max_context: int = 4096,
    system_prompt: Optional[str] = CONTEXT_EXTRACTION_SYSTEM_PROMPT,
    config: Optional[dict] = None,
    verbose: bool = False,
    context_selector: Optional[ContextSelector] = None,
) -> None:
    """
    Generates and saves a context extraction file for a given index note.
    
    Args:
        index_note: The index note to process.
        model: The LLM model object.
        info_notes: Optional list of specific notes to process.
        system_prompt: Optional system prompt override.
        context_selector: A function that determines which previous contexts 
                          should be included. Defaults to a sliding window of 5.
    """
    # 0. Set default selector if None
    if context_selector is None:
        context_selector = _default_context_selector

    # 1. Resolve the Master List (File Structure)
    all_associated_notes = _resolve_info_notes_for_index(index_note)
    
    # Create the ordered list of names for the selector
    all_note_names = [n.name for n in all_associated_notes]
    
    # 2. Determine the Work Queue
    target_notes_set = set(info_notes) if info_notes else set(all_associated_notes)

    if not all_associated_notes:
        print(f"Warning: No info notes found for '{index_note.name}'.")
        return

    # 3. Determine Context Note Path
    prefix = "_index_"
    interesting_name = index_note.name[len(prefix):] if index_note.name.startswith(prefix) else index_note.name
    context_note_name = f"_context_extraction_{interesting_name}"
    parent_dir = Path(index_note.rel_path).parent
    context_rel_path = parent_dir / f"{context_note_name}.md"
    
    context_note = VaultNote(index_note.vault, rel_path=str(context_rel_path))

    # 4. Load Existing Data
    existing_data = {}
    if context_note.exists():
        print(f"Found existing context note: {context_note.name}. Parsing...")
        existing_data = _parse_existing_context_note(context_note.text())
    else:
        context_note.create()

    # 5. Process Notes Sequentially
    print(f"Processing sequence of {len(all_associated_notes)} info note(s)...")
    
    current_data = existing_data.copy()
    
    for i, note in enumerate(all_associated_notes):
        # Determine if we should run the LLM for this note
        should_run_llm = (note in target_notes_set) and (note.name not in current_data)
        
        if should_run_llm:
            print(f"[{i+1}/{len(all_associated_notes)}] Extracting context for '{note.name}'...")
            
            # Prepare Text
            mf = MarkdownFile.from_vault_note(note)
            note_text = str(process_standard_information_note(mf, note.vault))
            
            # --- NEW LOGIC: Select Context ---
            # Pass the ordered list of names to the selector
            relevant_note_names = context_selector(note, current_data, all_note_names)
            
            selected_context_strings = []
            for name in relevant_note_names:
                # Double check existence, though selector should handle it
                if name in current_data:
                    selected_context_strings.append(current_data[name])
            
            prev_context_str = "\n".join(selected_context_strings)
            # ---------------------------------
            
            # Format Input
            llm_input = _format_llm_input(prev_context_str, note_text)
            
            # Run Prediction
            context_info = extract_context_with_lm(
                model, llm_input, max_context=max_context, system_prompt=system_prompt, config=config,verbose=verbose)
            
            if context_info:
                # Update Data
                current_data[note.name] = context_info
                
                # Write to file immediately
                full_content = _format_context_extraction(all_associated_notes, current_data)
                context_note.write(full_content)
            else:
                print(f"  -> No context returned for '{note.name}'.")

    print(f"Context extraction completed: {context_note.name}")

In [ ]:
import unittest
from unittest.mock import MagicMock, patch, call
from pathlib import Path

class TestContextExtraction(unittest.TestCase):

    def setUp(self):
        # Common Mocks
        self.mock_vault = MagicMock()
        self.mock_model = MagicMock()
        
        # Mock Index Note
        self.index_note = MagicMock()
        self.index_note.name = "_index_Algebra"
        self.index_note.rel_path = "Algebra/_index_Algebra.md"
        self.index_note.vault = self.mock_vault
        # Mock pathlib behavior for parent directory
        self.index_note.path = Path("Algebra/_index_Algebra.md")
        
        # Mock Info Notes
        self.note_a = MagicMock()
        self.note_a.name = "Definition A"
        self.note_a.text.return_value = "Content of A"
        
        self.note_b = MagicMock()
        self.note_b.name = "Theorem B"
        self.note_b.text.return_value = "Content of B"

    def test_format_llm_input(self):
        """
        Verifies that the previous context and new passage are combined correctly.
        """
        # Call the function directly from the global scope (__main__)
        prev = "K is a field."
        new_text = "Let V be a vector space over K."
        
        # Assuming _format_llm_input is defined in the notebook cells above
        result = _format_llm_input(prev, new_text)
        
        self.assertIn("PREVIOUS CONTEXT:\nK is a field.", result)
        self.assertIn("NEW PASSAGE:\nLet V be a vector space over K.", result)

    def test_parse_existing_context_note(self):
        """
        Verifies that the markdown parser correctly extracts note-context pairs.
        """
        sample_text = (
            "- [[Definition A]]\n"
            "    - Context for A line 1\n"
            "      Context for A line 2\n"
            "- [[Theorem B]]\n"
            "    - Context for B"
        )
        
        # Call the function directly
        result = _parse_existing_context_note(sample_text)
        
        self.assertEqual(result['Definition A'], "Context for A line 1\nContext for A line 2")
        self.assertEqual(result['Theorem B'], "Context for B")

    # Patching objects in __main__ because we are running inside the notebook
    @patch('__main__.VaultNote')
    @patch('__main__._resolve_info_notes_for_index')
    @patch('__main__.process_standard_information_note')
    @patch('__main__.MarkdownFile')
    @patch('__main__.extract_context_with_lm')
    @patch('__main__._format_context_extraction')
    def test_context_accumulation_flow(self, mock_format, mock_extract, mock_md_file, mock_process, mock_resolve, mock_vault_note_cls):
        """
        CRITICAL TEST: Verifies that context generated for Note A is passed 
        as 'Previous Context' when processing Note B.
        """
        # 1. Setup Dependencies
        mock_resolve.return_value = [self.note_a, self.note_b]
        
        # Mock the text processing of the notes
        mock_process.side_effect = ["Processed Text A", "Processed Text B"]
        
        # Mock the LLM responses
        # First call (Note A) returns "Context A"
        # Second call (Note B) returns "Context B"
        mock_extract.side_effect = ["Context A", "Context B"]
        
        # Mock the Context Note file creation
        mock_context_note = MagicMock()
        mock_context_note.exists.return_value = False # File doesn't exist yet
        mock_vault_note_cls.return_value = mock_context_note

        # 2. Execute
        create_context_extraction_for_index_note(self.index_note, self.mock_model)

        # 3. Verify LLM Calls
        # We expect 2 calls to the LLM.
        self.assertEqual(mock_extract.call_count, 2)
        
        # Check arguments for Note A (First Call)
        args_a, _ = mock_extract.call_args_list[0]
        input_text_a = args_a[1] # 0 is model, 1 is text
        self.assertIn("NEW PASSAGE:\nProcessed Text A", input_text_a)
        
        # Check arguments for Note B (Second Call)
        # CRITICAL: The input text MUST contain "Context A" (the result of the previous step)
        args_b, _ = mock_extract.call_args_list[1]
        input_text_b = args_b[1]
        self.assertIn("PREVIOUS CONTEXT:\nContext A", input_text_b)
        self.assertIn("NEW PASSAGE:\nProcessed Text B", input_text_b)

        # 4. Verify Write Calls
        # Should write twice (incremental saving)
        self.assertEqual(mock_context_note.write.call_count, 2)

    @patch('__main__.VaultNote')
    @patch('__main__._resolve_info_notes_for_index')
    @patch('__main__.process_standard_information_note')
    @patch('__main__.MarkdownFile')
    @patch('__main__.extract_context_with_lm')
    @patch('__main__._parse_existing_context_note')
    @patch('__main__._format_context_extraction')
    def test_incremental_update_uses_existing_context(self, mock_format, mock_parse, mock_extract, mock_md_file, mock_process, mock_resolve, mock_vault_note_cls):
        """
        Verifies that if Note A already has context in the file, we skip LLM for A,
        but still pass A's context to Note B.
        """
        # 1. Setup Dependencies
        mock_resolve.return_value = [self.note_a, self.note_b]
        
        # Mock existing data in the file
        mock_context_note = MagicMock()
        mock_context_note.exists.return_value = True
        mock_context_note.text.return_value = "Raw Markdown Content"
        mock_vault_note_cls.return_value = mock_context_note
        
        # Parser returns existing context for Note A
        mock_parse.return_value = {self.note_a.name: "Existing Context A"}
        
        # Mock processing for Note B (Note A shouldn't be processed)
        mock_process.return_value = "Processed Text B"
        
        # Mock LLM for Note B
        mock_extract.return_value = "New Context B"

        # 2. Execute
        create_context_extraction_for_index_note(self.index_note, self.mock_model)

        # 3. Verify Logic
        # LLM should be called ONLY ONCE (for Note B)
        self.assertEqual(mock_extract.call_count, 1)
        
        # Verify the input to that single call
        args, _ = mock_extract.call_args
        input_text = args[1]
        
        # CRITICAL: Even though we didn't run LLM on Note A, its "Existing Context A"
        # must be present in the prompt for Note B.
        self.assertIn("PREVIOUS CONTEXT:\nExisting Context A", input_text)
        self.assertIn("NEW PASSAGE:\nProcessed Text B", input_text)
        
        # Verify we wrote the file (updating with B)
        self.assertTrue(mock_context_note.write.called)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.006s

OK


Processing sequence of 2 info note(s)...
[1/2] Extracting context for 'Definition A'...
[2/2] Extracting context for 'Theorem B'...
Context extraction completed: <MagicMock name='VaultNote().name' id='2024840544176'>
Found existing context note: <MagicMock name='VaultNote().name' id='2024794194096'>. Parsing...
Processing sequence of 2 info note(s)...
[2/2] Extracting context for 'Theorem B'...
Context extraction completed: <MagicMock name='VaultNote().name' id='2024794194096'>


In [ ]:
#| export
def get_context_extraction_map(vault, reference: str) -> Dict[str, str]:
    """
    Locates and parses the context extraction note for a given reference.
    
    Args:
        vault: The Vault object.
        reference: The reference string (e.g., 'Algebra').
        
    Returns:
        A dictionary mapping note names to their extracted context strings.
        Returns an empty dict if the context note does not exist.
    """
    # Construct the expected name of the context extraction note
    context_note_name = f"_context_extraction_{reference}"
    
    # Find the note in the vault
    context_note = VaultNote(vault, name=context_note_name)
    
    if not context_note.exists():
        # print(f"Debug: Context extraction note '{context_note_name}' not found.")
        return {}
        
    # Use the existing helper to parse the markdown content
    return _parse_existing_context_note(context_note.text())

def get_accumulated_context_for_notes(
    info_notes: List[VaultNote], 
    reference: str
) -> str:
    """
    Retrieves and accumulates the context extraction text for a list of notes.
    
    This function looks up the context for each note in the provided list
    and joins them into a single string, suitable for passing to an LLM.
    
    Args:
        info_notes: The list of VaultNote objects to retrieve context for.
        reference: The reference string used to locate the context file.
        
    Returns:
        A single string containing the combined context for the found notes,
        separated by double newlines.
    """
    if not info_notes:
        return ""
        
    vault = info_notes[0].vault
    
    # 1. Get the master dictionary of context data
    context_map = get_context_extraction_map(vault, reference)
    
    accumulated_parts = []
    
    # 2. Iterate through the requested notes in order
    for note in info_notes:
        if note.name in context_map:
            context_text = context_map[note.name]
            # Only add non-empty context
            if context_text.strip():
                accumulated_parts.append(context_text)
            
    return "\n\n".join(accumulated_parts)